# Quellenübergreifender Abgleich und Analyse
In den ersten drei Notebooks haben wir die Soll-Daten (CIS) sowie die beiden Ist-Quellen (EMR, PRC) jeweils einzeln aufbereitet und als bereinigte Dateien abgelegt. Hier führen wir sie zusammen. Zuvor verschaffen wir uns einen Überblick über alle drei Quellen - Umfang, Spalten, eindeutige Fahrzeuge, Fahrer, Touren und Zeitraum, um zu sehen, wo bereits eine gemeinsame, vergleichbare Grundlage besteht und wo wir sie erst herstellen müssen.

Der gemeinsame Fahrzeug-Schlüssel ist `LKW_KENNZ`. Schon vorab ist klar, dass die Quellen nicht in allen Dimensionen deckungsgleich sind: Einen Fahrernamen führen nur Soll und EMR, einen Tour-Schlüssel führen Soll (`TOURNR`) und PRC (`TourID`/`StationID`, noch mit Suffix/`TP`), während EMR keinen nativen Tour-Schlüssel besitzt.

In [2]:
import pandas as pd
from pathlib import Path

PFAD_PROCESSED = Path.cwd().parent / "data" / "processed"

# Pro Quelle: Datei, Datumsspalten und die für den Abgleich relevanten Schlüsselspalten
quellen = {
    "Soll (CIS)": {"datei": "solldaten_cis_clean.csv",
                   "datumsspalten": ["STARTDATUM", "ANKUNFT", "ABFAHRT"],
                   "tour": "TOURNR", "kennz": "LKW_KENNZ", "fahrer": "FAHRERNAME", "zeit": "STARTDATUM"},
    "Ist (EMR)":  {"datei": "istdaten_emr_clean.csv",
                   "datumsspalten": ["Meldungszeit"],
                   "tour": None, "kennz": "LKW_KENNZ", "fahrer": "FAHRERNAME", "zeit": "Meldungszeit"},
    "Ist (PRC)":  {"datei": "istdaten_prc_clean.csv",
                   "datumsspalten": ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug", "Zeitstempel_Server"],
                   "tour": "TourID", "kennz": "LKW_KENNZ", "fahrer": None, "zeit": "Zeitstempel_Fahrzeug"},
}

# Einlesen (Datumsspalten direkt parsen, da CSV die Typen nicht erhält)
dfs = {name: pd.read_csv(PFAD_PROCESSED / c["datei"], parse_dates=c["datumsspalten"])
       for name, c in quellen.items()}

# Vergleichsübersicht
zeilen = []
for name, df in dfs.items():
    c = quellen[name]
    zeilen.append({
        "Quelle": name,
        "Zeilen": len(df),
        "Spalten": df.shape[1],
        "LKW_KENNZ": df[c["kennz"]].nunique(),
        "Fahrer": df[c["fahrer"]].nunique() if c["fahrer"] else None,
        "Touren (nativ)": f"{df[c['tour']].nunique()} ({c['tour']})" if c["tour"] else "—",
        "Zeitraum von": df[c["zeit"]].min(),
        "Zeitraum bis": df[c["zeit"]].max(),
    })
uebersicht = pd.DataFrame(zeilen).set_index("Quelle")
display(uebersicht)

# Spalten je Quelle
for name, df in dfs.items():
    print(f"\n{name} ({df.shape[1]} Spalten):")
    print(" ", list(df.columns))

,Zeilen,Spalten,LKW_KENNZ,Fahrer,Touren (nativ),Zeitraum von,Zeitraum bis
Quelle,,,,,,,
Soll (CIS),1699,17,12,13.0,203 (TOURNR),2026-03-02 05:00:00,2026-03-31 06:00:00
Ist (EMR),150967,10,12,13.0,—,2026-03-02 00:00:00,2026-03-31 23:56:00
Ist (PRC),29430,15,12,NaN,200 (TourID),2026-03-01 00:00:00,2026-03-31 13:25:19



Soll (CIS) (17 Spalten):
  ['TOURNR', 'NUMMER', 'TOURSTATION_ID', 'TYP', 'BEZEICHNUNG', 'TOURENART', 'STARTDATUM', 'ANKUNFT', 'ABFAHRT', 'NAME1', 'PLZ', 'ORT', 'STRASSE', 'GEOX', 'GEOY', 'LKW_KENNZ', 'FAHRERNAME']

Ist (EMR) (10 Spalten):
  ['FAHRERNAME', 'Formularnummer', 'Formularbeschreibung', 'Meldungszeit', 'Geschwindigkeit', 'KM-Stand', 'Breitengrad', 'Längengrad', 'Position', 'LKW_KENNZ']

Ist (PRC) (15 Spalten):
  ['msg_typ', 'PositionsID', 'Longitude', 'Latitude', 'Geschwindigkeit', 'KMStand', 'Zeitstempel_GPS', 'Zeitstempel_Fahrzeug', 'Zeitstempel_Server', 'LKW_KENNZ', 'Status', 'TourID', 'StationID', 'SendposID', 'Ereignis_Typ']


### Spaltennamen vereinheitlichen
Die drei Quellen benennen gleiche Konzepte unterschiedlich. Wir bringen sie auf eine einheitliche Konvention. Gleiche Konzepte erhalten identische Namen, quellenspezifische Spalten werden nur an die Konvention angepasst. Geschwindigkeit, Kilometerstand und die Geokoordinaten heißen danach überall gleich. Die vom Fahrzeug gemeldete Zeit wird in EMR und PRC einheitlich MELDUNGSZEIT, während PRCs GPS- und Server-Zeitstempel zur Unterscheidung erhalten bleiben. Den Tour-Schlüssel vereinheitlichen wir hier bewusst noch nicht, da Soll TOUR_NR, PRC TOUR_ID und STATION_ID führt.

In [3]:
# Umbenennungstabelle je Quelle: gleiche Konzepte -> identische Namen,
# quellenspezifische Spalten nur an die Konvention (GROSS_SNAKE, deutsch, ohne Umlaute) angepasst.
umbenennung = {
    "Soll (CIS)": {
        "TOURNR": "TOUR_NR", "GEOX": "LAENGENGRAD", "GEOY": "BREITENGRAD",
    },
    "Ist (EMR)": {
        "Formularnummer": "FORMULARNUMMER", "Formularbeschreibung": "FORMULARBESCHREIBUNG",
        "Meldungszeit": "MELDUNGSZEIT", "Geschwindigkeit": "GESCHWINDIGKEIT",
        "KM-Stand": "KM_STAND", "Breitengrad": "BREITENGRAD", "Längengrad": "LAENGENGRAD",
        "Position": "POSITION",
    },
    "Ist (PRC)": {
        "msg_typ": "MELDUNGSTYP", "PositionsID": "POSITIONS_ID",
        "Longitude": "LAENGENGRAD", "Latitude": "BREITENGRAD",
        "Geschwindigkeit": "GESCHWINDIGKEIT", "KMStand": "KM_STAND",
        "Zeitstempel_GPS": "ZEITSTEMPEL_GPS", "Zeitstempel_Fahrzeug": "MELDUNGSZEIT",
        "Zeitstempel_Server": "ZEITSTEMPEL_SERVER", "Status": "STATUS",
        "TourID": "TOUR_ID", "StationID": "STATION_ID", "SendposID": "SENDPOS_ID",
        "Ereignis_Typ": "EREIGNIS_TYP",
    },
}

for name, mapping in umbenennung.items():
    dfs[name] = dfs[name].rename(columns=mapping)

# Kontrolle: Spalten je Quelle und die gemeinsamen Konzept-Spalten
for name, df in dfs.items():
    print(f"{name} ({df.shape[1]} Spalten):")
    print(" ", list(df.columns))
print()
for konzept in ["LKW_KENNZ", "FAHRERNAME", "BREITENGRAD", "LAENGENGRAD",
                "GESCHWINDIGKEIT", "KM_STAND", "MELDUNGSZEIT"]:
    print(f"  {konzept}: {[name for name, df in dfs.items() if konzept in df.columns]}")

Soll (CIS) (17 Spalten):
  ['TOUR_NR', 'NUMMER', 'TOURSTATION_ID', 'TYP', 'BEZEICHNUNG', 'TOURENART', 'STARTDATUM', 'ANKUNFT', 'ABFAHRT', 'NAME1', 'PLZ', 'ORT', 'STRASSE', 'LAENGENGRAD', 'BREITENGRAD', 'LKW_KENNZ', 'FAHRERNAME']
Ist (EMR) (10 Spalten):
  ['FAHRERNAME', 'FORMULARNUMMER', 'FORMULARBESCHREIBUNG', 'MELDUNGSZEIT', 'GESCHWINDIGKEIT', 'KM_STAND', 'BREITENGRAD', 'LAENGENGRAD', 'POSITION', 'LKW_KENNZ']
Ist (PRC) (15 Spalten):
  ['MELDUNGSTYP', 'POSITIONS_ID', 'LAENGENGRAD', 'BREITENGRAD', 'GESCHWINDIGKEIT', 'KM_STAND', 'ZEITSTEMPEL_GPS', 'MELDUNGSZEIT', 'ZEITSTEMPEL_SERVER', 'LKW_KENNZ', 'STATUS', 'TOUR_ID', 'STATION_ID', 'SENDPOS_ID', 'EREIGNIS_TYP']

  LKW_KENNZ: ['Soll (CIS)', 'Ist (EMR)', 'Ist (PRC)']
  FAHRERNAME: ['Soll (CIS)', 'Ist (EMR)']
  BREITENGRAD: ['Soll (CIS)', 'Ist (EMR)', 'Ist (PRC)']
  LAENGENGRAD: ['Soll (CIS)', 'Ist (EMR)', 'Ist (PRC)']
  GESCHWINDIGKEIT: ['Ist (EMR)', 'Ist (PRC)']
  KM_STAND: ['Ist (EMR)', 'Ist (PRC)']
  MELDUNGSZEIT: ['Ist (EMR)', 'Ist (PR

## Gemeinsamer Tour-Schlüssel
Soll führt `TOUR_NR`, PRC trägt die Tournummer in `TOUR_ID` und `STATION_ID`, EMR besitzt gar keinen Tour-Schlüssel. Bevor wir etwas zusammenführen, müssen wir die in PRC vorhandene Tournummer sauber herausziehen und dabei eine Besonderheit berücksichtigen.

In manchen `STATION_ID` steht nicht nur `H/42375-TP3`, sondern z. B. `H/42375_2-TP4_2`. Auf Nachfrage hat Frank das als nachträgliche Umplanung im TMS eingeordnet: Wird eine Tour während der Ausführung geändert, erhalten die betroffenen oder neuen Stopps ein Versions-Suffix (`_2`, `_3`) an der Tournummer und/oder am `TP`, damit die bereits abgearbeiteten Stopps unberührt bleiben. Das ist eine Arbeitshypothese, passt aber zu dem, was die TMS-Screenshots zeigen - eine `Aktualisierung` mitten in der laufenden Tour `H/42375`.

Daraus ergibt sich unsere Behandlung: `H/42375` und `H/42375_2` gehören zur selben physischen Tour, nur zu verschiedenen Planungsständen. Für die Zuordnung und das Zählen von Touren ist also die **Basis-Tournummer** (ohne Suffix) der Schlüssel. Gleichzeitig dürfen wir die Suffixe und das `TP` nicht verwerfen - sie sind die Umplanung selbst und damit genau das, was die Routenabweichungsanalyse später untersucht. Wir brauchen daher beides nebeneinander: eine Basis-Tournummer zum Verbinden und die vollständige Original-Information zum Analysieren.

Bevor wir die Basis-Tournummer ableiten, sehen wir uns zunächst faktisch an, welche Formate in `TOUR_ID` und `STATION_ID` tatsächlich vorkommen. So legen wir das Extraktionsmuster auf Basis der echten Daten fest und nicht auf Basis einzelner Beispiele.

### Sichtung der Tour- und Stationsformate
Wir abstrahieren jeden Wert auf sein Strukturmuster (Buchstaben-Läufe als `L`, Ziffern-Läufe als `D`, Trennzeichen bleiben erhalten) und zählen, welche Muster vorkommen. So wird das Suffix-Schema sichtbar, ohne dass wir es annehmen.

In [4]:
import re

def struktur(wert):
    """Abstrahiert einen Wert auf sein Format: Buchstaben-Läufe -> L, Ziffern-Läufe -> D,
    Trennzeichen bleiben. So werden die vorkommenden Formate sichtbar, nicht die Einzelwerte."""
    if pd.isna(wert):
        return None
    s = re.sub(r"[A-Za-z]+", "L", wert)
    s = re.sub(r"\d+", "D", s)
    return s

prc = dfs["Ist (PRC)"]

for spalte in ["TOUR_ID", "STATION_ID"]:
    werte = prc[spalte].dropna()
    strukt = werte.map(struktur)
    print(f"=== {spalte}: {werte.nunique()} eindeutige Werte, {strukt.nunique()} Strukturmuster ===")
    uebersicht = (pd.DataFrame({"struktur": strukt.values, "wert": werte.values})
                  .groupby("struktur")
                  .agg(zeilen=("wert", "size"),
                       eindeutige_werte=("wert", "nunique"),
                       beispiel=("wert", "first"))
                  .sort_values("zeilen", ascending=False))
    display(uebersicht)
    print()

=== TOUR_ID: 200 eindeutige Werte, 1 Strukturmuster ===


,zeilen,eindeutige_werte,beispiel
struktur,,,
L/D,1244,200,H/42380



=== STATION_ID: 1526 eindeutige Werte, 3 Strukturmuster ===


,zeilen,eindeutige_werte,beispiel
struktur,,,
L/D-LD,6009,1229,H/42379-TP1
L/D_D-LD_D,1154,246,H/42378_2-TP1_2
L/D_D-LD,233,51,H/42451_2-TP1


### Basis-Tournummer ableiten
Die Sichtung zeigt: Das Versions-Suffix sitzt immer an der Tournummer (manchmal zusätzlich am `TP`), nie nur am `TP`. Die Basis-Tournummer ist daher robust als Präfix plus erste Ziffernfolge zu extrahieren — also alles bis vor dem ersten `_` oder `-`. Wir ziehen sie aus `TOUR_ID` bzw. `STATION_ID` (je nach Meldungstyp trägt nur eines davon die Tour) in eine neue Spalte `TOUR_BASIS`. Die vollständige Original-`STATION_ID` mit Suffix und `TP` bleibt erhalten, da sie die Umplanung abbildet, die wir später analysieren. Zusätzlich halten wir fest, bei wie vielen Basis-Touren überhaupt eine Re-Disposition vorkommt, und sehen uns das Format von Soll `TOUR_NR` an, um den anschließenden Abgleich vorzubereiten.

In [5]:
# Basis-Tournummer: Präfix + erste Ziffernfolge, vor einem etwaigen Versions-Suffix/TP-Teil.
# Je Meldung trägt nur TOUR_ID (Tourstatus) oder STATION_ID (Stationsstatus) die Tour.
prc["TOUR_BASIS"] = prc["TOUR_ID"].fillna(prc["STATION_ID"]).str.extract(r"^([A-Za-z]+/\d+)")[0]

print("PRC – Basis-Tournummern")
print("  eindeutige Basis-Touren:", prc["TOUR_BASIS"].nunique())
umgeplant = prc.loc[prc["STATION_ID"].str.contains("_", na=False), "TOUR_BASIS"].nunique()
print("  davon mit Re-Disposition (mind. eine suffixbehaftete Station):", umgeplant)
print("  Beispiele:", sorted(prc["TOUR_BASIS"].dropna().unique())[:5])

soll = dfs["Soll (CIS)"]
print("\nSoll – TOUR_NR (zur Vorbereitung des Abgleichs)")
print("  eindeutige TOUR_NR:", soll["TOUR_NR"].nunique())
print("  Strukturmuster:", soll["TOUR_NR"].dropna().map(struktur).value_counts().to_dict())
print("  Beispiele:", sorted(soll["TOUR_NR"].dropna().unique())[:5])

PRC – Basis-Tournummern
  eindeutige Basis-Touren: 200
  davon mit Re-Disposition (mind. eine suffixbehaftete Station): 71
  Beispiele: ['H/42374', 'H/42375', 'H/42376', 'H/42377', 'H/42378']

Soll – TOUR_NR (zur Vorbereitung des Abgleichs)
  eindeutige TOUR_NR: 203
  Strukturmuster: {'L/D': 1699}
  Beispiele: ['H/42374', 'H/42375', 'H/42376', 'H/42377', 'H/42378']


### Abgleich der Tourmengen Soll ↔ PRC
Die Sichtung hat bestätigt, dass beide Schlüssel dasselbe Format haben (`H/<Nummer>`) und sich direkt vergleichen lassen. Jetzt halten wir die Basis-Tournummern aus PRC gegen die `TOUR_NR` aus Soll und sehen, wie groß die Überschneidung wirklich ist: welche Touren in beiden Quellen vorkommen, welche nur geplant wurden (Soll ohne PRC) und welche nur in der Telematik auftauchen (PRC ohne Soll). Daraus ergibt sich, wie tragfähig die gemeinsame Grundlage ist und welche Touren wir gesondert betrachten müssen.

In [6]:
soll_touren = set(dfs["Soll (CIS)"]["TOUR_NR"].dropna().unique())
prc_touren  = set(prc["TOUR_BASIS"].dropna().unique())

gemeinsam = soll_touren & prc_touren
nur_soll  = soll_touren - prc_touren
nur_prc   = prc_touren  - soll_touren

print(f"Touren in Soll:          {len(soll_touren)}")
print(f"Touren in PRC:           {len(prc_touren)}")
print(f"In beiden Quellen:       {len(gemeinsam)}")
print(f"Nur in Soll (kein PRC):  {len(nur_soll)}  -> {sorted(nur_soll)}")
print(f"Nur in PRC (kein Soll):  {len(nur_prc)}  -> {sorted(nur_prc)}")

Touren in Soll:          203
Touren in PRC:           200
In beiden Quellen:       200
Nur in Soll (kein PRC):  3  -> ['H/42416', 'H/42429', 'H/42452']
Nur in PRC (kein Soll):  0  -> []


Der Abgleich ist nahezu vollständig: Alle 200 PRC-Touren finden sich in Soll wieder, es gibt keine einzige Tour, die nur in der Telematik auftaucht. Die Tournummer ist damit als gemeinsamer Schlüssel zwischen Soll und PRC belastbar. Lediglich drei geplante Touren (H/42416, H/42429, H/42452) haben keine PRC-Daten. Diese kleine Differenz sehen wir uns einzeln an, um den Grund zu verstehen, statt sie nur zu zählen.

### Die drei Touren ohne PRC genauer ansehen
Drei geplante Touren haben keine PRC-Daten. Statt sie pauschal abzuhaken, sehen wir uns für jede an, welches Fahrzeug und welcher Tag dahinterstehen, und prüfen gegen, ob das Fahrzeug an diesem Tag überhaupt PRC-Meldungen geliefert hat. Daraus ergibt sich der Grund: keine Telematik an dem Tag, nur GPS-Pings ohne Statusmeldungen, oder ein anderer Tourverlauf. Der Grund bestimmt, wie wir die Touren im weiteren Vergleich behandeln.

In [7]:
nur_soll_touren = ["H/42416", "H/42429", "H/42452"]
soll = dfs["Soll (CIS)"]

for t in nur_soll_touren:
    st = soll[soll["TOUR_NR"] == t]
    kennz = st["LKW_KENNZ"].dropna().unique()
    tage = sorted(pd.unique(
        pd.concat([st["STARTDATUM"], st["ANKUNFT"], st["ABFAHRT"]]).dropna().dt.date))
    print("=" * 70)
    print(f"{t}: Fahrzeug {list(kennz)}, {len(st)} Stationen, Tag(e) {tage}")
    print(f"   Soll-Zeitfenster: {st['ANKUNFT'].min()}  bis  {st['ABFAHRT'].max()}")
    for k in kennz:
        pt = prc[(prc["LKW_KENNZ"] == k) & (prc["MELDUNGSZEIT"].dt.date.isin(tage))]
        print(f"   PRC für {k} an diesen Tagen: {len(pt)} Meldungen")
        if len(pt):
            print("     nach Meldungstyp:", pt["MELDUNGSTYP"].value_counts().to_dict())
            print("     PRC-Basis-Touren an dem Tag:", sorted(pt["TOUR_BASIS"].dropna().unique()))

H/42416: Fahrzeug ['PI-EN-900'], 4 Stationen, Tag(e) [datetime.date(2026, 3, 2)]
   Soll-Zeitfenster: 2026-03-02 10:00:00  bis  2026-03-02 11:14:00
   PRC für PI-EN-900 an diesen Tagen: 0 Meldungen
H/42429: Fahrzeug ['PI-EN-900'], 4 Stationen, Tag(e) [datetime.date(2026, 3, 6)]
   Soll-Zeitfenster: 2026-03-06 06:00:00  bis  2026-03-06 07:14:00
   PRC für PI-EN-900 an diesen Tagen: 18 Meldungen
     nach Meldungstyp: {'Position': 16, 'Tourstatus': 1, 'FMSStatus': 1}
     PRC-Basis-Touren an dem Tag: ['H/42445']
H/42452: Fahrzeug ['OD-TS-8046'], 3 Stationen, Tag(e) [datetime.date(2026, 3, 9)]
   Soll-Zeitfenster: 2026-03-09 10:00:00  bis  2026-03-09 11:53:00
   PRC für OD-TS-8046 an diesen Tagen: 21 Meldungen
     nach Meldungstyp: {'Position': 21}
     PRC-Basis-Touren an dem Tag: []


Alle drei Touren sind aufgeklärt, jede zeigt ein anderes Muster:

- H/42416 (PI-EN-900, 02.03.): keine PRC-Meldungen an dem Tag - das Fahrzeug hat keine Telematik gesendet, die Tour ist in PRC nicht abgebildet.
- H/42429 (PI-EN-900, 06.03.): das Fahrzeug hat gesendet, aber für eine andere Tour (H/42445). Die geplante Tour wurde an dem Tag nicht in dieser Form ausgeführt.
- H/42452 (OD-TS-8046, 09.03.): nur Position-Pings, keine Status- oder Tourmeldungen - das Fahrzeug war unterwegs, hat aber keine Stationsstati geliefert.

Keine der drei ist ein Fehler der Aufbereitung. Zwei sind Telematik-Lücken (gar nicht gesendet bzw. nur GPS), eine ist eine tatsächliche Tourabweichung, daher bleiben alle drei in Soll erhalten.

### Ereignis-Paletten von PRC und EMR sichten
EMR und PRC beschreiben dieselben physischen Ereignisse (Ankunft, Beladebeginn, Abfahrt, Tourbeginn/-ende …), aber in unterschiedlicher Codierung: PRC trägt einen Statuscode, dessen Bedeutung vom Meldungstyp abhängt; EMR trägt eine Formularnummer mit Klartext-Beschreibung. Bevor wir beide über das C-Logistic-Mapping auf gemeinsame Klartext-Ereignisse abbilden, sehen wir uns zuerst faktisch an, welche Ereignisse in jeder Quelle überhaupt vorkommen und wie häufig. Wir hinterlegen je (Meldungstyp, Status)-Kombination die Bedeutung aus den C-Logistic-Unterlagen, damit die Palette ohne Nachschlagen lesbar ist. Die Bedeutung hängt am Paar aus Meldungstyp und Status, nicht am Status allein. Für die Statuscodes 12–15 (Ankunft, Beginn/Ende Laden/Entladen, Abfahrt) stimmen Mapping-XML und Schnittstellenbeschreibung überein. Status 11 ist dagegen uneindeutig: Das HTS-Mapping bildet darauf das SPEDION-Ereignis „angenommen" ab, die generische Schnittstellenbeschreibung nennt 11 „Anfahrt". Das markieren wir als offen, statt es festzulegen - für die spätere Ereignis-Brücke (Ankunft/Abfahrt/Laden) wird Status 11 ohnehin nicht benötigt. Position, FMSStatus und Ereignis tragen keinen Status; bei Ereignis steckt die Bedeutung in EREIGNIS_TYP.

In [8]:
# Bedeutung je (MELDUNGSTYP, STATUS) aus den C-Logistic-Unterlagen.
# 12-15 in beiden Quellen einheitlich; 11 dort widersprüchlich -> bewusst als offen markiert.
status_bedeutung = {
    ("Stationsstatus", 3):  "Station abgelehnt/gelöscht",
    ("Stationsstatus", 11): "angenommen / Anfahrt (Doku uneindeutig)",
    ("Stationsstatus", 12): "Ankunft",
    ("Stationsstatus", 13): "Beginn Laden/Entladen",
    ("Stationsstatus", 14): "Ende Laden/Entladen",
    ("Stationsstatus", 15): "Abfahrt",
    ("Tourstatus", 4):  "Fehler/Storno",
    ("Tourstatus", 11): "Tour gestartet",
    ("Tourstatus", 14): "Tour beendet",
    ("Tourstatus", 22): "empfangen/übertragen (administrativ)",
    ("Tourstatus", 28): "Aktualisierung empfangen (administrativ)",
    ("Sendposstatus", 11): "Geladen",
    ("Sendposstatus", 12): "Entladen",
}
typ_ohne_status = {
    "Position":  "GPS-Position (kein Status)",
    "FMSStatus": "Fahrzeugtelematik (kein Status)",
    "Ereignis":  "Ereignis (Bedeutung über EREIGNIS_TYP)",
}

def status_text(row):
    if pd.isna(row["STATUS"]):
        return typ_ohne_status.get(row["MELDUNGSTYP"], "—")
    return status_bedeutung.get((row["MELDUNGSTYP"], int(row["STATUS"])), "unbekannt")

print("PRC – vorkommende (MELDUNGSTYP, STATUS)-Kombinationen mit Bedeutung:")
prc_palette = (prc.groupby(["MELDUNGSTYP", "STATUS"], dropna=False)
                  .size().reset_index(name="anzahl")
                  .sort_values(["MELDUNGSTYP", "STATUS"]))
prc_palette["BEDEUTUNG"] = prc_palette.apply(status_text, axis=1)
display(prc_palette)

PRC – vorkommende (MELDUNGSTYP, STATUS)-Kombinationen mit Bedeutung:


,MELDUNGSTYP,STATUS,anzahl,BEDEUTUNG
0,Ereignis,NaN,2,Ereignis (Bedeutung über EREIGNIS_TYP)
1,FMSStatus,NaN,6677,Fahrzeugtelematik (kein Status)
2,Position,NaN,11189,GPS-Position (kein Status)
3,Sendposstatus,11.0,1548,Geladen
4,Sendposstatus,12.0,1374,Entladen
5,Stationsstatus,3.0,31,Station abgelehnt/gelöscht
6,Stationsstatus,11.0,1497,angenommen / Anfahrt (Doku uneindeutig)
7,Stationsstatus,12.0,1478,Ankunft
8,Stationsstatus,13.0,1471,Beginn Laden/Entladen
9,Stationsstatus,14.0,1460,Ende Laden/Entladen


In [9]:
emr = dfs["Ist (EMR)"]
print("EMR – vorkommende (FORMULARNUMMER, FORMULARBESCHREIBUNG)-Kombinationen:")
emr_palette = (emr.groupby(["FORMULARNUMMER", "FORMULARBESCHREIBUNG"], dropna=False)
                  .size().reset_index(name="anzahl")
                  .sort_values("anzahl", ascending=False))
display(emr_palette)

EMR – vorkommende (FORMULARNUMMER, FORMULARBESCHREIBUNG)-Kombinationen:


,FORMULARNUMMER,FORMULARBESCHREIBUNG,anzahl
65,30019,Rollen,43173
19,1644,Beginn Rollen,25088
18,1643,Ende Rollen,25068
66,30020,Anhalten,23515
3,1337,Place wurde aktiviert,8803
...,...,...,...
23,1685,Warten,1
32,1840,Tankfüllstand Event - Große Abnahme,1
51,10070,Android Update installed by MDM successful,1
42,8112,Navigation - Automatisch angezeigte Alternativ...,1


In [10]:
pd.set_option("display.max_rows", None)
display(emr_palette.sort_values("FORMULARNUMMER"))
pd.reset_option("display.max_rows")

,FORMULARNUMMER,FORMULARBESCHREIBUNG,anzahl
0,1156,Fahren ohne Fahrerkarte,46
1,1332,Tour im Fahrzeug angekommen,239
2,1334,Place wurde gelesen,263
3,1337,Place wurde aktiviert,8803
4,1344,"Tour, Place oder Order im Fahrzeug gelöscht",17
5,1345,"Storno (Tour, Place oder Order) nicht verarbeitet",1
6,1352,Tourdetails hinzufügen zum Fahrzeug(z.B. Place...,83
7,1370,Update verarbeitet,117
8,1380,Place wurde nicht am korrekten Ort abgearbeitet,125
9,1382,TourDeleteByID - OK,17


### Kern-Mapping der physischen Ereignisse
Wir definieren eine Tabelle, die die physischen Kernereignisse beschreibt, die in beiden Ist-Quellen ein Pendant haben, und bilden sie auf gemeinsame Ereignisnamen ab: die EMR-Formularnummer und die PRC-(Meldungstyp, Status)-Kombination zeigen auf dasselbe Ereignis. Das ist eine reine Definitionstabelle, noch keine Zuordnung einzelner Zeilen.

Bewusst aufgenommen sind nur die abgesicherten Anker: Ankunft, Abfahrt, Ladebeginn und Tour-Empfang. Laden und Entladen führen wir getrennt (EMR unterscheidet sie über die Formulare 1751/1753), während PRC beides in Stationsstatus 13 zusammenfasst - das halten wir als Hinweis fest, statt Information zu verlieren. Nicht aufgenommen sind Ereignisse ohne klares Gegenstück: PRCs „Ende Laden/Entladen" (Status 14) und Tourbeginn/-ende haben in EMR kein sauberes Pendant, und Status 11 ist in der Doku uneindeutig.

Anschließend kontrollieren wir, wie viele Zeilen je Quelle einem Ankerereignis zugeordnet werden, und stellen die Häufigkeiten gegenüber. Dass nur ein kleiner Teil der Zeilen ein Anker ist, ist erwartet: Der Großteil sind Bewegungs- und Systemmeldungen (EMR) bzw. GPS- und Telematikmeldungen (PRC), die nicht zu den Stationsereignissen gehören.

In [11]:
ereignis_mapping = pd.DataFrame([
    ("ANKUNFT",         1650, "Ankunft Kunde",               "Stationsstatus", 12, ""),
    ("ABFAHRT",         1651, "Abfahrt Kunde",               "Stationsstatus", 15, ""),
    ("BEGINN_LADEN",    1751, "Laden beginnen",              "Stationsstatus", 13, "PRC trennt Laden/Entladen nicht"),
    ("BEGINN_ENTLADEN", 1753, "Entladen beginnen",           "Stationsstatus", 13, "PRC trennt Laden/Entladen nicht"),
    ("TOUR_EMPFANGEN",  1332, "Tour im Fahrzeug angekommen", "Tourstatus",     22, "administrativ, Tour-Ebene"),
], columns=["EREIGNIS", "EMR_FORMULARNUMMER", "EMR_BESCHREIBUNG",
            "PRC_MELDUNGSTYP", "PRC_STATUS", "HINWEIS"])
display(ereignis_mapping)

,EREIGNIS,EMR_FORMULARNUMMER,EMR_BESCHREIBUNG,PRC_MELDUNGSTYP,PRC_STATUS,HINWEIS
0,ANKUNFT,1650,Ankunft Kunde,Stationsstatus,12,
1,ABFAHRT,1651,Abfahrt Kunde,Stationsstatus,15,
2,BEGINN_LADEN,1751,Laden beginnen,Stationsstatus,13,PRC trennt Laden/Entladen nicht
3,BEGINN_ENTLADEN,1753,Entladen beginnen,Stationsstatus,13,PRC trennt Laden/Entladen nicht
4,TOUR_EMPFANGEN,1332,Tour im Fahrzeug angekommen,Tourstatus,22,"administrativ, Tour-Ebene"


In [12]:
# EMR: Formularnummer -> Ereignis
emr_zu_ereignis = {1650: "ANKUNFT", 1651: "ABFAHRT",
                   1751: "BEGINN_LADEN", 1753: "BEGINN_ENTLADEN", 1332: "TOUR_EMPFANGEN"}
emr["EREIGNIS"] = emr["FORMULARNUMMER"].map(emr_zu_ereignis)

# PRC: (Meldungstyp, Status) -> Ereignis; Status 13 bleibt kombiniert, da PRC nicht trennt
prc_zu_ereignis = {("Stationsstatus", 12): "ANKUNFT", ("Stationsstatus", 15): "ABFAHRT",
                   ("Stationsstatus", 13): "BEGINN_LADEN_ENTLADEN", ("Tourstatus", 22): "TOUR_EMPFANGEN"}
def prc_ereignis(row):
    if pd.isna(row["STATUS"]):
        return pd.NA
    return prc_zu_ereignis.get((row["MELDUNGSTYP"], int(row["STATUS"])), pd.NA)
prc["EREIGNIS"] = prc.apply(prc_ereignis, axis=1)

# Kontrolle
print("EMR – Zeilen mit Ankerereignis:", emr["EREIGNIS"].notna().sum(), "von", len(emr))
print("PRC – Zeilen mit Ankerereignis:", prc["EREIGNIS"].notna().sum(), "von", len(prc))
print()
vergleich = pd.DataFrame({"EMR": emr["EREIGNIS"].value_counts(),
                          "PRC": prc["EREIGNIS"].value_counts()})
display(vergleich)

EMR – Zeilen mit Ankerereignis: 4183 von 150967
PRC – Zeilen mit Ankerereignis: 4626 von 29430



,EMR,PRC
EREIGNIS,,
ABFAHRT,1322.0,1459.0
ANKUNFT,1302.0,1478.0
BEGINN_ENTLADEN,785.0,NaN
BEGINN_LADEN,535.0,NaN
BEGINN_LADEN_ENTLADEN,NaN,1471.0
TOUR_EMPFANGEN,239.0,218.0


Die echten Häufigkeiten bestätigen die Brücke: Ankunft (1.302 ↔ 1.478), Abfahrt (1.322 ↔ 1.459) und Tour-Empfang (239 ↔ 218) liegen je in derselben Größenordnung. Die Lade-/Entlade-Summe aus EMR (535 + 785 = 1.320) passt zur kombinierten PRC-Zahl (1.471). Die PRC-Werte liegen durchgängig etwas höher - PRC erfasst die Stationsereignisse einen Tick vollständiger als EMR. Dieser konsistente Versatz bestätigt, dass die Ereignisse dasselbe Geschehen beschreiben, und unterstreicht zugleich, dass die Brücke als Anker pro Ereignis dient und nicht als exakter Mengenabgleich. Nur rund 4.000 der Zeilen je Quelle sind Ankerereignisse; der große Rest sind Bewegungs-, Telematik- und Systemmeldungen, die nicht zu den Stationsereignissen zählen.

### Zeitliche Nähe der Anker prüfen
Bevor wir eine zeitbasierte Zuordnung bauen, sehen wir uns faktisch an, wie nah die gleichartigen Ankerereignisse beider Quellen zeitlich beieinanderliegen. Dazu wählen wir ein Fahrzeug an einem Tag mit vielen Ankern in beiden Quellen und legen die EMR- und PRC-Ankerereignisse rein chronologisch nebeneinander — ohne sie zu paaren. So wird unmittelbar sichtbar, ob eine EMR-Ankunft zeitlich dicht an einer PRC-Ankunft desselben Fahrzeugs liegt. Daraus erkennen wir, ob eine Zuordnung über ein enges Zeitfenster trägt oder ob die Zeiten zu weit auseinanderlaufen. Die Tour kennen wir dabei nur über die PRC-Seite — genau das ist der Hebel, über den EMR später eine Tour bekommt.

In [13]:
emr_anker = emr[emr["EREIGNIS"].notna()].copy()
prc_anker = prc[prc["EREIGNIS"].notna()].copy()
for d in (emr_anker, prc_anker):
    d["DATUM"] = d["MELDUNGSZEIT"].dt.date

# Beispiel-Fahrzeug/-Tag wählen: das mit den meisten Ankern in BEIDEN Quellen
ez = emr_anker.groupby(["LKW_KENNZ", "DATUM"]).size()
pz = prc_anker.groupby(["LKW_KENNZ", "DATUM"]).size()
gemeinsam = pd.DataFrame({"EMR": ez, "PRC": pz}).dropna()
kennz, tag = gemeinsam.assign(minimum=gemeinsam.min(axis=1)).sort_values("minimum", ascending=False).index[0]
print(f"Beispiel: {kennz} am {tag}  "
      f"(EMR-Anker: {int(gemeinsam.loc[(kennz, tag), 'EMR'])}, "
      f"PRC-Anker: {int(gemeinsam.loc[(kennz, tag), 'PRC'])})")

# EMR- und PRC-Anker dieses Fahrzeugs/Tags rein chronologisch nebeneinander
e = (emr_anker[(emr_anker["LKW_KENNZ"] == kennz) & (emr_anker["DATUM"] == tag)]
     .assign(QUELLE="EMR", TOUR=pd.NA))
p = (prc_anker[(prc_anker["LKW_KENNZ"] == kennz) & (prc_anker["DATUM"] == tag)]
     .assign(QUELLE="PRC", TOUR=lambda x: x["TOUR_BASIS"]))
zeitachse = (pd.concat([e[["QUELLE", "MELDUNGSZEIT", "EREIGNIS", "TOUR"]],
                        p[["QUELLE", "MELDUNGSZEIT", "EREIGNIS", "TOUR"]]])
               .sort_values("MELDUNGSZEIT"))
display(zeitachse)

Beispiel: WL-PL-431 am 2026-03-09  (EMR-Anker: 56, PRC-Anker: 59)


,QUELLE,MELDUNGSZEIT,EREIGNIS,TOUR
65086,EMR,2026-03-09 05:36:12,TOUR_EMPFANGEN,<NA>
6136,PRC,2026-03-09 05:36:12,TOUR_EMPFANGEN,H/42480
6216,PRC,2026-03-09 06:05:49,BEGINN_LADEN_ENTLADEN,H/42480
6214,PRC,2026-03-09 06:05:49,ANKUNFT,H/42480
65117,EMR,2026-03-09 06:05:49,BEGINN_LADEN,<NA>
...,...,...,...,...
7367,PRC,2026-03-09 14:10:06,ANKUNFT,H/42480
7369,PRC,2026-03-09 14:10:06,BEGINN_LADEN_ENTLADEN,H/42480
7373,PRC,2026-03-09 14:10:07,ABFAHRT,H/42480
66225,EMR,2026-03-09 16:58:39,TOUR_EMPFANGEN,<NA>


Die gleichartigen Anker liegen nicht nur nah beieinander, sie haben exakt denselben Zeitstempel (auf die Sekunde): EMR TOUR_EMPFANGEN 05:36:12 ↔ PRC 05:36:12, die Stationsereignisse um 06:05:49 ↔ 06:05:49, der zweite Tour-Empfang 16:58:39 ↔ 16:58:39. Wir brauchen kein unsicheres Toleranzfenster. EMR-Meldungen lassen sich über (Fahrzeug, Zeitstempel) praktisch exakt an die PRC-Ereignisse — und damit an die Tour — anbinden.

EMR und PRC sind dann keine unabhängigen Ist-Quellen, die sich gegenseitig bestätigen, sondern zwei Formate desselben Stroms. Sie sind komplementär in der Abdeckung (EMR trägt die feinen Bewegungs-/Systemereignisse, PRC die saubere Tour-/Stationsstruktur), nicht voneinander unabhängig.

### Hält die zeitliche Übereinstimmung über alle Anker?
Das Beispiel zeigte eine sekundengenaue Übereinstimmung - bevor wir darauf eine Zuordnung aufbauen, prüfen wir, ob das über alle Anker hält und nicht nur an diesem einen Tag. Dazu ordnen wir jedem EMR-Anker den zeitlich nächsten PRC-Anker desselben Fahrzeugs zu (über merge_asof, Richtung „nächster", begrenzt auf ein großzügiges 10-Minuten-Fenster) und betrachten die Verteilung der absoluten Zeitdifferenz. Liegt der Großteil bei null oder wenigen Sekunden, bestätigt das, dass EMR und PRC denselben Ereignisstrom abbilden — und die spätere Anbindung von EMR an die PRC-Tour wird über (Fahrzeug, Zeit) praktisch verlustfrei. Wir merken den PRC-Zeitstempel in einer eigenen Spalte vor, damit wir die Differenz überhaupt messen können; das großzügige Fenster dient hier nur der Diagnose, nicht der späteren Zuordnung.

In [14]:
emr_a = emr_anker[["LKW_KENNZ", "MELDUNGSZEIT", "EREIGNIS"]].sort_values("MELDUNGSZEIT")
prc_a = prc_anker[["LKW_KENNZ", "MELDUNGSZEIT", "EREIGNIS", "TOUR_BASIS"]].sort_values("MELDUNGSZEIT").copy()
prc_a["PRC_ZEIT"] = prc_a["MELDUNGSZEIT"]   # eigene Spalte, um die Differenz messen zu können

abgleich = pd.merge_asof(emr_a, prc_a, on="MELDUNGSZEIT", by="LKW_KENNZ",
                         direction="nearest", tolerance=pd.Timedelta("10min"),
                         suffixes=("_EMR", "_PRC"))
abgleich["delta_sek"] = (abgleich["MELDUNGSZEIT"] - abgleich["PRC_ZEIT"]).abs().dt.total_seconds()

n = len(abgleich)
ohne = abgleich["PRC_ZEIT"].isna().sum()
print(f"EMR-Anker gesamt: {n}")
print(f"  ohne PRC-Anker im 10-min-Fenster: {ohne}")
print(f"  delta == 0 s:        {(abgleich['delta_sek'] == 0).sum()}")
print(f"  delta <= 1 s:        {(abgleich['delta_sek'] <= 1).sum()}")
print(f"  delta <= 60 s:       {(abgleich['delta_sek'] <= 60).sum()}")
print(f"  delta > 60 s:        {(abgleich['delta_sek'] > 60).sum()}")

EMR-Anker gesamt: 4183
  ohne PRC-Anker im 10-min-Fenster: 44
  delta == 0 s:        3425
  delta <= 1 s:        3964
  delta <= 60 s:       4094
  delta > 60 s:        45
